# 🌳 CART Decision Tree — Wine Quality Regression

Systematic comparative analysis using `DecisionTreeRegressor` (**CART**) trained on three different
partitions (70/30, 40/60, 80/20) to assess the predictive model's robustness, with **train and test**
evaluation on each split.

| | |
|---|---|
| **Predictor variables (X)** | `alcohol`, `volatile acidity`, `sulphates` |
| **Target variable (Y)** | `quality` |
| **Model** | `DecisionTreeRegressor` (**CART**, scikit-learn) |
| **Validation strategy** | Three splits: 70/30 · 40/60 · 80/20 (train + test) |
| **Overfitting handling** | Regularization (`max_depth`, `min_samples_leaf`) + oversampling of rare `quality` values with `resample` (train only) |

> **Dataset:** UCI Wine Quality — 1,599 red wine samples with physicochemical properties and a sensory score.

> **Note on oversampling in regression:** unlike classification, the target here is numeric, so there is no
> binary "minority class". The `quality` distribution is nonetheless heavily imbalanced toward the central
> values (5 and 6), so we oversample the **rare quality values** (the extremes, e.g. 3, 4, 8) in the training
> set with `sklearn.utils.resample`. This is the regression analogue of minority oversampling and helps the
> tree pay attention to under-represented quality levels. The main lever against overfitting in regression
> remains **regularization** (`max_depth`, `min_samples_leaf`).

## I. Initial Setup — Imports and Visual Style

In [ ]:
# ── CELL 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

import warnings
warnings.filterwarnings('ignore')

# Consistent visual style across the project
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print("✅ Libraries imported successfully")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")
print(f"   seaborn {sns.__version__}")

## II. Loading Data from UCI

We load the `winequality-red.csv` dataset directly from the UCI repository. We use `alcohol`,
`volatile acidity`, and `sulphates` as predictors (X) and `quality` as the target (Y), following the
feature selection from the linear regression notebook based on Spearman correlation.

In [ ]:
# ── CELL 2: Load data from UCI ────────────────────────────────────────────────
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

wine_quality = pd.read_csv(URL, sep=";")

# Variables selected by highest Spearman correlation with quality
FEATURES = ['alcohol', 'volatile acidity', 'sulphates']
TARGET   = 'quality'

print(f"✅ Data loaded from UCI")
print(f"   Rows    : {wine_quality.shape[0]:,}")
print(f"   Columns : {wine_quality.shape[1]}")
print(f"\n📋 Missing values:")
print(wine_quality[FEATURES + [TARGET]].isnull().sum())
print(f"\n📋 quality distribution (note the imbalance toward 5 and 6):")
print(wine_quality[TARGET].value_counts().sort_index())
wine_quality[FEATURES + [TARGET]].head()

In [ ]:
# ── CELL 3: Prepare X / Y vectors ─────────────────────────────────────────────
X = wine_quality[FEATURES].copy()
y = wine_quality[TARGET].copy()

print(f"X shape: {X.shape}  (predictors: {FEATURES})")
print(f"y shape: {y.shape}  (target: {TARGET})")
print(f"\n📊 Descriptive statistics:")
wine_quality[FEATURES + [TARGET]].describe().round(4)

## III. Oversampling Rare `quality` Values with `resample`

The `quality` target is dominated by values 5 and 6, while extreme values (3, 4, 8) are rare. To prevent
the tree from ignoring those under-represented levels (and overfitting to the central mass), we oversample
the rare quality values in the **training set only** using `sklearn.utils.resample`.

> **Key rule:** oversampling is applied **only to the training set**, after the split. The test set keeps
> its real distribution so the evaluation stays honest.

In [ ]:
# ── CELL 4: Oversampling helper for regression target (train only) ────────────
def oversample_rare_quality(X_train, y_train, random_state=42):
    """
    Oversample rare quality values in the TRAINING set only, using
    sklearn.utils.resample. Each quality level is upsampled (with
    replacement) to the size of the most frequent level, so all
    quality levels become roughly balanced. Returns (X_bal, y_bal).
    """
    train = X_train.copy()
    train['__target__'] = y_train.values

    # Size of the most frequent quality level
    counts = train['__target__'].value_counts()
    n_max  = counts.max()

    parts = []
    for level, grp in train.groupby('__target__'):
        if len(grp) < n_max:
            grp = resample(grp, replace=True, n_samples=n_max,
                           random_state=random_state)
        parts.append(grp)

    balanced = pd.concat(parts).sample(frac=1, random_state=random_state)
    y_bal = balanced['__target__']
    X_bal = balanced.drop(columns='__target__')
    return X_bal, y_bal

print("✅ `oversample_rare_quality` defined — applied to TRAIN only (resample)")
print("   All quality levels upsampled to the most frequent level's size.")

## IV. Logic Block — Per-Split Evaluation Function

We define a reusable function that trains, evaluates (on **train and test**), and visualizes the **CART**
`DecisionTreeRegressor` for any data partition. Two regularization levers (`max_depth`, `min_samples_leaf`)
are exposed to control overfitting, and the rare-quality oversampling is applied to the training set only.

In [ ]:
# ── CELL 5: Reusable evaluation function ──────────────────────────────────────
def evaluate_split(X, y, test_size, split_label, global_results,
                   max_depth=8, min_samples_leaf=20, use_oversample=True):
    """
    Train a CART DecisionTreeRegressor, compute metrics on TRAIN and TEST,
    and produce two plots: residuals and predicted vs actual.

    Parameters
    ----------
    X, y             : predictors / target
    test_size        : fraction for test (e.g. 0.20)
    split_label      : label for titles
    global_results   : list collecting per-split results
    max_depth        : regularization — limits tree depth (overfitting control)
    min_samples_leaf : regularization — min samples per leaf
    use_oversample   : if True, oversample rare quality values on TRAIN only
    """
    train_pct = int((1 - test_size) * 100)
    test_pct  = int(test_size * 100)

    # ── 1. Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    # ── 1b. Oversample rare quality values — TRAIN ONLY
    if use_oversample:
        X_train_use, y_train_use = oversample_rare_quality(X_train, y_train, 42)
    else:
        X_train_use, y_train_use = X_train, y_train

    print(f"\n{'='*58}")
    print(f"  {split_label} — {train_pct}% Train / {test_pct}% Test")
    print(f"{'='*58}")
    print(f"  Train (original): {X_train.shape[0]:,}  |  "
          f"Train (oversampled): {X_train_use.shape[0]:,}  |  "
          f"Test: {X_test.shape[0]:,}")

    # ── 2. Train (regularized CART)
    model = DecisionTreeRegressor(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    model.fit(X_train_use, y_train_use)

    # ── 3. Predictions (train + test)
    y_pred_train = model.predict(X_train_use)
    y_pred       = model.predict(X_test)

    # ── 4. Metrics (train + test)
    r2_tr   = r2_score(y_train_use, y_pred_train)
    r2      = r2_score(y_test, y_pred)
    mse     = mean_squared_error(y_test, y_pred)
    rmse    = np.sqrt(mse)
    mae     = mean_absolute_error(y_test, y_pred)

    print(f"\n  📊 METRICS ({split_label})")
    print(f"  {'─'*45}")
    print(f"  R² Train : {r2_tr:.4f}   |   R² Test : {r2:.4f}   "
          f"(gap = {r2_tr - r2:.4f})")
    print(f"  MSE  (Mean Squared Error) : {mse:.4f}")
    print(f"  RMSE (Root MSE)           : {rmse:.4f}  (in quality points)")
    print(f"  MAE  (Mean Absolute Error): {mae:.4f}  (in quality points)")
    print(f"\n  🌳 Tree structure:")
    print(f"  Max depth   : {model.get_depth()}")
    print(f"  Leaves      : {model.get_n_leaves()}")
    print(f"  Total nodes : {model.tree_.node_count}")

    # Store for consolidated table
    global_results.append({
        'Split':    split_label,
        'Train%':   train_pct,
        'Test%':    test_pct,
        'R2_Train': round(r2_tr, 4),
        'R2_Test':  round(r2,    4),
        'MSE':      round(mse,   4),
        'RMSE':     round(rmse,  4),
        'MAE':      round(mae,   4),
        'Leaves':   model.get_n_leaves(),
    })

    # ── 5. Plot 1 — Residuals ──────────────────────────────────
    residuals = y_test.values - y_pred

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'CART DecisionTreeRegressor — {split_label}',
                 fontsize=14, fontweight='bold')

    axes[0].scatter(y_pred, residuals, alpha=0.35, s=14, color='#4ecdc4',
                    label='Residuals')
    axes[0].axhline(0, color='#ff6b6b', linestyle='--', linewidth=2,
                    label='Ideal line (error = 0)')
    order_idx  = np.argsort(y_pred)
    lowess_res = sm.nonparametric.lowess(
        residuals[order_idx], y_pred[order_idx], frac=0.4
    )
    axes[0].plot(lowess_res[:, 0], lowess_res[:, 1],
                 color='#ffa07a', linewidth=2.5, label='LOWESS trend')
    axes[0].set_title('Residual Plot (Errors)', fontweight='bold')
    axes[0].set_xlabel('Predicted values (quality)')
    axes[0].set_ylabel('Residuals (actual − predicted quality)')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # ── Plot 2 — Predicted vs Actual ───────────────────────────
    minval = min(y_test.min(), y_pred.min())
    maxval = max(y_test.max(), y_pred.max())
    axes[1].scatter(y_test, y_pred, alpha=0.30, s=14, color='#95e1d3',
                    label='Predictions')
    axes[1].plot([minval, maxval], [minval, maxval],
                 color='#ff6b6b', linewidth=2, linestyle='--',
                 label='Perfect prediction')
    axes[1].set_title('Predicted vs Actual', fontweight='bold')
    axes[1].set_xlabel('Actual value (quality)')
    axes[1].set_ylabel('Predicted value (quality)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    return model, y_test, y_pred, X_train_use, y_train_use


# Results container for the consolidated analysis
global_results = []
print("✅ evaluate_split defined.")
print(f"   Predictors : {FEATURES}")
print(f"   Target     : {TARGET}")
print("   CART regularized (max_depth, min_samples_leaf) + resample on TRAIN")

---
## Split 1 — 70% Train / 30% Test

A moderate partition. Larger test set for a more representative error estimate.

In [ ]:
# ── CELL 6: Split 1 — 70/30 ───────────────────────────────────────────────────
model_s1, y_test_s1, y_pred_s1, X_train_s1, y_train_s1 = evaluate_split(
    X, y,
    test_size=0.30,
    split_label='Split 1 (70/30)',
    global_results=global_results,
)

---
## Split 2 — 40% Train / 60% Test

The most demanding partition. With less training data this tests whether the tree generalizes well
or tends to overfit.

In [ ]:
# ── CELL 7: Split 2 — 40/60 ───────────────────────────────────────────────────
model_s2, y_test_s2, y_pred_s2, X_train_s2, y_train_s2 = evaluate_split(
    X, y,
    test_size=0.60,
    split_label='Split 2 (40/60)',
    global_results=global_results,
)

---
## Split 3 — 80% Train / 20% Test

The most common standard partition. Maximizes training data, usually giving the best possible fit.

In [ ]:
# ── CELL 8: Split 3 — 80/20 ───────────────────────────────────────────────────
model_s3, y_test_s3, y_pred_s3, X_train_s3, y_train_s3 = evaluate_split(
    X, y,
    test_size=0.20,
    split_label='Split 3 (80/20)',
    global_results=global_results,
)

---
## III. Consolidated Analysis (Summary)

We compare the three splits' performance (train vs test), analyze the residual implications, and issue a
recommendation on whether to further refine the tree or scale to ensemble methods.

In [ ]:
# ── CELL 9: Comparative metrics table ─────────────────────────────────────────
df_summary = pd.DataFrame(global_results)

print("=" * 72)
print("        COMPARATIVE TABLE — THREE SPLITS (train + test)")
print("=" * 72)
print(df_summary.to_string(index=False))
print("=" * 72)

In [ ]:
# ── CELL 10: Comparative metrics chart per split ──────────────────────────────
metrics = ['R2_Test', 'RMSE', 'MAE', 'R2_Train']
colors  = ['#ff6b6b', '#4ecdc4', '#95e1d3', '#ffa07a']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Metric Comparison — CART DecisionTreeRegressor (3 splits) — Wine Quality',
             fontsize=13, fontweight='bold', y=1.01)

for ax, (metric, color) in zip(axes.flat, zip(metrics, colors)):
    values = df_summary[metric].values
    labels = [s.replace('Split ', '').replace('(', '').replace(')', '')
              for s in df_summary['Split'].values]
    bars   = ax.bar(labels, values, color=color, edgecolor='white', width=0.5)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel('Value')
    ax.set_ylim(0, max(values) * 1.2 + 1e-6)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values) * 0.03,
                f'{val:.4f}', ha='center', va='bottom',
                fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 11: Interpretive analysis and recommendation ─────────────────────────
r2_test_vals  = df_summary['R2_Test'].values
r2_train_vals = df_summary['R2_Train'].values
rmse_vals     = df_summary['RMSE'].values

r2_range   = r2_test_vals.max()   - r2_test_vals.min()
rmse_range = rmse_vals.max() - rmse_vals.min()

print("=" * 68)
print("   CONSOLIDATED ANALYSIS — CART DecisionTreeRegressor — Wine Quality")
print("=" * 68)

print("""
1. PERFORMANCE VARIATION ACROSS SPLITS
─────────────────────────────────────""")
print(f"   R² Test  max={r2_test_vals.max():.4f}  min={r2_test_vals.min():.4f}  "
      f"range={r2_range:.4f}")
print(f"   RMSE     max={rmse_vals.max():.4f}  min={rmse_vals.min():.4f}  "
      f"range={rmse_range:.4f}")

if r2_range < 0.05:
    print("""
   ✅ R² is stable across the three splits (variation < 0.05).
      The model generalizes well and does not critically depend
      on the size of the training partition.""")
else:
    print("""
   ⚠️  R² varies noticeably across splits (variation >= 0.05).
      The model is sensitive to the training set size, a possible
      sign of tree overfitting.""")

print("""
2. OVERFITTING DIAGNOSIS (Train vs Test R² per split)
──────────────────────────────────────────────────────""")
for _, row in df_summary.iterrows():
    gap = row['R2_Train'] - row['R2_Test']
    if gap > 0.10:
        flag = "❌ severe"
    elif gap > 0.05:
        flag = "⚠️ moderate"
    else:
        flag = "✅ healthy"
    print(f"   {row['Split']:<16}  R²Train={row['R2_Train']:.4f}  "
          f"R²Test={row['R2_Test']:.4f}  gap={gap:+.4f}  {flag}")

print("""
3. RESIDUAL IMPLICATIONS
────────────────────────
   • Vertical bands in the residual plot are normal for a decision
     tree: each leaf produces a fixed prediction, generating groups
     of points at the same height.
   • Regularization (max_depth, min_samples_leaf) plus rare-quality
     oversampling reduces the number of leaves and the train-test gap
     compared to an unconstrained tree.""")

best_idx   = df_summary['R2_Test'].idxmax()
best_split = df_summary.loc[best_idx, 'Split']
best_r2    = df_summary.loc[best_idx, 'R2_Test']
best_rmse  = df_summary.loc[best_idx, 'RMSE']
print(f"\n   Best split: {best_split}  ->  R²Test={best_r2:.4f}, RMSE={best_rmse:.4f} pts")

print("""
4. RECOMMENDATION
─────────────────
   A) TREE REFINEMENT (already applied here):
      • Limit depth      : DecisionTreeRegressor(max_depth=4..8)
      • Regularize leaves: min_samples_leaf=10..50
      • Search optimal hyperparameters with GridSearchCV.
      • Oversample rare quality values on TRAIN with resample.

   CONCLUSION: a regularized CART tree controls the overfitting that an
   unconstrained tree shows on this data. The discrete nature of quality
   (only a few integer values) limits the maximum achievable R². If R²
   is stable across splits and the train-test gap is small, the model is
   reliable; further gains would come from ensemble methods or richer
   feature engineering.""")
print("=" * 68)

---
## IV. Decision Tree Visualization

To understand the prediction mechanism, we visualize:

1. **Real trained tree (truncated)** — the first levels of `model_s3` (Split 80/20).
2. **Decision rules in text** — explicit reading of the tree paths.

In [ ]:
# ── CELL 12: Visualization of the real tree (first 4 levels) ──────────────────
from sklearn.tree import plot_tree, export_text

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    model_s3,
    max_depth=4,
    feature_names=FEATURES,
    filled=True,
    rounded=True,
    impurity=False,
    precision=2,
    fontsize=9,
    ax=ax,
)
ax.set_title(
    'CART Decision Tree — Split 80/20 (first 4 depth levels)\n'
    f'Total depth: {model_s3.get_depth()}  |  '
    f'Leaves: {model_s3.get_n_leaves()}  |  '
    f'Nodes: {model_s3.tree_.node_count}',
    fontsize=13, fontweight='bold', pad=15
)
plt.tight_layout()
plt.show()

print(f"\n📐 Tree information (model_s3, regularized):")
print(f"   Max depth   : {model_s3.get_depth()}")
print(f"   Leaves      : {model_s3.get_n_leaves()}")
print(f"   Total nodes : {model_s3.tree_.node_count}")
print(f"\n   Feature used at the root node:")
root_feature = FEATURES[model_s3.tree_.feature[0]]
root_thresh  = model_s3.tree_.threshold[0]
print(f"   → {root_feature} <= {root_thresh:.4f}")

In [ ]:
# ── CELL 13: Decision rules in text ───────────────────────────────────────────
print("=" * 62)
print("  DECISION RULES — CART Tree (Split 80/20, regularized)")
print("=" * 62)
rules = export_text(model_s3, feature_names=FEATURES, max_depth=3)
print(rules)

print("=" * 62)
print("📖 HOW TO READ THE TREE:")
print("  |--- variable <= X.XX  -> left branch (condition TRUE)")
print("  |--- variable >  X.XX  -> right branch (condition FALSE)")
print("  |--- value: [Y.YY]     -> predicted quality at that leaf")
print()
print("📋 CHEMICAL INTERPRETATION:")
print("  • Splits on 'alcohol'          -> higher alcohol = higher quality")
print("  • Splits on 'volatile acidity' -> higher acidity = lower quality")
print("  • Splits on 'sulphates'        -> higher sulphates = higher quality")
print("  These rules are consistent with the EDA and Spearman correlation.")
print("=" * 62)

---
## 🎓 Final Conclusions

| Aspect | Finding |
|---|---|
| **Model** | Regularized **CART** `DecisionTreeRegressor` (`max_depth`, `min_samples_leaf`) controls the overfitting that an unconstrained tree shows on discrete `quality`. |
| **Splits** | Evaluated on 70/30, 40/60, and 80/20, reporting **train and test** R² to expose the overfitting gap directly. |
| **Oversampling** | Rare `quality` values are oversampled with `resample` on the **training set only**; the test set keeps its real distribution. |
| **Metrics** | R², MSE, RMSE, and MAE computed and interpreted per split. |
| **Residuals** | Vertical bands due to the tree's discrete predictions (≠ horizontal bands of linear regression). |
| **Root variable** | `alcohol` — the tree's first split, confirming it is the most informative predictor. |
| **vs Linear Regression** | The tree captures non-linear relationships; expected R² above the linear baseline. |
| **Next step** | Tune hyperparameters with `GridSearchCV` for a finer regularization trade-off. |

> **Note:** This notebook complements the linear regression analyses, adding the perspective of
> non-parametric tree-based models for the UCI Wine Quality dataset.